In [1]:
!pip install lightgbm -q

import numpy as np
import pandas as pd
import pickle
import os
import time
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.preprocessing import MinMaxScaler
from sktime.transformations.panel.rocket import MiniRocketMultivariate
from sklearn.linear_model import LogisticRegressionCV
import gc
import warnings
warnings.filterwarnings('ignore')

print("✅ 导入完成")

✅ 导入完成


In [2]:
def temporal_cutout(X, y, cut_length=100, cut_channels=None):
    """Temporal Cutout: 随机遮盖一段连续时间和部分通道"""
    X_aug = X.copy()
    n_samples, length, n_channels = X.shape
    
    if cut_channels is None:
        cut_channels = np.random.choice(
            n_channels, 
            size=np.random.randint(int(n_channels*0.3), int(n_channels*0.7)),
            replace=False
        )
    
    for i in range(n_samples):
        start = np.random.randint(0, length - cut_length)
        X_aug[i, start:start+cut_length, cut_channels] = 0
    
    return X_aug, y


def temporal_mixup(X, y, alpha=0.2):
    """Temporal Mixup: 混合两个样本"""
    n_samples = len(X)
    indices = np.random.permutation(n_samples)
    lam = np.random.beta(alpha, alpha)
    X_mixed = lam * X + (1 - lam) * X[indices]
    y_mixed = lam * y + (1 - lam) * y[indices]
    return X_mixed, y_mixed


def temporal_cutmix(X, y, cut_length=100):
    """Temporal Cutmix: 用另一个样本的时间片段替换当前样本"""
    X_aug = X.copy()
    n_samples, length, n_channels = X.shape
    indices = np.random.permutation(n_samples)
    
    for i in range(n_samples):
        start = np.random.randint(0, length - cut_length)
        X_aug[i, start:start+cut_length, :] = X[indices[i], start:start+cut_length, :]
    
    return X_aug, y


def apply_augmentation(X, y, aug_type='cutout', probability=0.5):
    """以一定概率应用数据增强"""
    if np.random.random() > probability:
        return X, y
    if aug_type == 'cutout':
        return temporal_cutout(X, y)
    elif aug_type == 'mixup':
        return temporal_mixup(X, y)
    elif aug_type == 'cutmix':
        return temporal_cutmix(X, y)
    return X, y

print("✅ 数据增强函数定义完成")

✅ 数据增强函数定义完成


In [3]:
data_dir = '/root'

with open(os.path.join(data_dir, 'flight_data.pkl'), 'rb') as f:
    data = pickle.load(f)

header_df = pd.read_csv(os.path.join(data_dir, 'flight_header.csv'))

print(f"✅ 数据加载成功")
print(f"  航班数: {len(data)}")

✅ 数据加载成功
  航班数: 11446


In [4]:
# 筛选完整19类基准子集
mask = (abs(header_df['date_diff']) <= 2) & (header_df['date_diff'] != 0)
mask = mask & (header_df['label'].notna())

filtered_header = header_df[mask].copy()
filtered_header = filtered_header.reset_index(drop=True)
flight_ids = filtered_header['Master Index'].values

print(f"📊 筛选后航班数: {len(filtered_header)}")
print(f"  维护后 (0): {sum(filtered_header['before_after']==0)}")
print(f"  维护前 (1): {sum(filtered_header['before_after']==1)}")

# 准备特征
target_len = 4096
X_list = []
y = []

print("\n⏳ 准备数据...")

for idx, flight_id in enumerate(flight_ids):
    sensor_data = data[flight_id]
    sensor_data = np.nan_to_num(sensor_data, nan=0.0)
    
    if sensor_data.shape[0] >= target_len:
        sensor_data = sensor_data[-target_len:, :]
    else:
        pad_width = ((0, target_len - sensor_data.shape[0]), (0, 0))
        sensor_data = np.pad(sensor_data, pad_width, mode='constant', constant_values=0)
    
    X_list.append(sensor_data)
    label = filtered_header.iloc[idx]['before_after']
    y.append(label)

X = np.array(X_list, dtype=np.float32)
y = np.array(y)

# 释放内存
del data, header_df, filtered_header, flight_ids, X_list
gc.collect()

print(f"\n✅ 数据准备完成")
print(f"  X 形状: {X.shape}")
print(f"  标签分布: 0={sum(y==0)}, 1={sum(y==1)}")

📊 筛选后航班数: 11446
  维护后 (0): 5844
  维护前 (1): 5602

⏳ 准备数据...

✅ 数据准备完成
  X 形状: (11446, 4096, 23)
  标签分布: 0=5844, 1=5602


In [5]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 减少核数量以节省内存
minirocket = MiniRocketMultivariate(
    random_state=42,
    num_kernels=3000,  # 进一步减少
)

classifier = LogisticRegressionCV(
    Cs=10, 
    cv=3, 
    random_state=42, 
    max_iter=5000
)

AUG_TYPES = ['cutout', 'mixup', 'cutmix']
AUG_PROBABILITY = 0.5

accuracies, f1_scores, auc_scores = [], [], []

print("\n" + "="*60)
print("5折交叉验证（数据增强）")
print("="*60)

fold = 1
for train_idx, val_idx in skf.split(X, y):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    print(f"\n--- Fold {fold} ---")
    print(f"  训练集: {len(train_idx)} 样本")
    print(f"  验证集: {len(val_idx)} 样本")
    
    start = time.time()
    
    # fold内归一化
    n_samples, length, channels = X_train.shape
    X_train_flat = X_train.reshape(-1, channels)
    scaler = MinMaxScaler()
    scaler.fit(X_train_flat)
    
    X_train_norm = scaler.transform(X_train_flat).reshape(n_samples, length, channels)
    X_val_flat = X_val.reshape(-1, channels)
    X_val_norm = scaler.transform(X_val_flat).reshape(X_val.shape)
    
    # 释放临时变量
    del X_train_flat, X_val_flat, scaler
    gc.collect()
    
    # 应用数据增强
    aug_type = np.random.choice(AUG_TYPES)
    X_train_aug, y_train_aug = apply_augmentation(
        X_train_norm, y_train, 
        aug_type=aug_type, 
        probability=AUG_PROBABILITY
    )
    
    # 释放原始训练数据（增强后不再需要）
    del X_train_norm
    gc.collect()
    
    # MiniRocket 特征提取
    X_train_transform = minirocket.fit_transform(X_train_aug, y_train_aug)
    X_val_transform = minirocket.transform(X_val_norm)
    
    # 释放增强数据
    del X_train_aug
    gc.collect()
    
    # 训练分类器
    classifier.fit(X_train_transform, y_train_aug)
    
    # 释放训练特征
    del X_train_transform, y_train_aug
    gc.collect()
    
    # 预测
    y_pred = classifier.predict(X_val_transform)
    y_prob = classifier.predict_proba(X_val_transform)[:, 1]
    
    # 释放验证特征
    del X_val_transform, X_val_norm
    gc.collect()
    
    elapsed = time.time() - start
    
    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    auc = roc_auc_score(y_val, y_prob)
    
    accuracies.append(acc)
    f1_scores.append(f1)
    auc_scores.append(auc)
    
    print(f"  增强方式: {aug_type}")
    print(f"  准确率: {acc:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}, 耗时: {elapsed:.2f}s")
    fold += 1

print("\n" + "="*60)
print("📊 数据增强实验最终结果:")
print("="*60)
print(f"  准确率: {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}")
print(f"  F1分数: {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")
print(f"  AUC:    {np.mean(auc_scores):.4f} ± {np.std(auc_scores):.4f}")
print("="*60)

print("\n📊 实验对比:")
print("-" * 60)
print(f"  无数据增强:              58.7%")
print(f"  数据增强 (Cutout/Mixup/Cutmix): {np.mean(accuracies):.4f}")
print("-" * 60)


5折交叉验证（数据增强）

--- Fold 1 ---
  训练集: 9156 样本
  验证集: 2290 样本
  增强方式: cutmix
  准确率: 0.5729, F1: 0.5485, AUC: 0.6068, 耗时: 75.83s

--- Fold 2 ---
  训练集: 9157 样本
  验证集: 2289 样本
  增强方式: cutmix
  准确率: 0.5950, F1: 0.5777, AUC: 0.6313, 耗时: 59.00s

--- Fold 3 ---
  训练集: 9157 样本
  验证集: 2289 样本
  增强方式: cutmix
  准确率: 0.5789, F1: 0.5590, AUC: 0.6081, 耗时: 62.23s

--- Fold 4 ---
  训练集: 9157 样本
  验证集: 2289 样本
  增强方式: cutmix
  准确率: 0.5998, F1: 0.5771, AUC: 0.6437, 耗时: 71.62s

--- Fold 5 ---
  训练集: 9157 样本
  验证集: 2289 样本
  增强方式: cutmix
  准确率: 0.5767, F1: 0.5557, AUC: 0.6066, 耗时: 56.20s

📊 数据增强实验最终结果:
  准确率: 0.5847 ± 0.0107
  F1分数: 0.5636 ± 0.0118
  AUC:    0.6193 ± 0.0154

📊 实验对比:
------------------------------------------------------------
  无数据增强:              58.7%
  数据增强 (Cutout/Mixup/Cutmix): 0.5847
------------------------------------------------------------
